<a href="https://colab.research.google.com/github/selvamaran/a26cnW2D2_timeseries/blob/main/Real_data_Spectral_Analysis_introduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [9]:
# @title Install
!pip install remfile h5py --quiet

In [10]:
# @title Imports
import ipywidgets as widgets
from IPython.display import display

import remfile, h5py
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, fftfreq


## Load IBL Neuropixels Data — Streams directly from DANDI (run me first!)
Streams 2 seconds from subject DY-008, channel 60 (best LFP+spike channel). No large download needed — remfile fetches only the bytes we ask for.

In [11]:
url  = 'https://api.dandiarchive.org/api/assets/604b7457-430e-4160-b7ff-594562d942e3/download/'
f    = h5py.File(remfile.File(url), 'r')
ap   = f['acquisition']['ElectricalSeriesProbe00AP']

In [12]:
ap_fs  = 30000.0       # Hz, confirmed from timestamps
conv   = 2.34375e-06   # to volts
BEST_CH = 60           # channel with clearest LFP structure + spikes
T_DUR   = 2.0          # seconds to stream
i1      = int(T_DUR * ap_fs)
t_ap    = np.arange(i1) / ap_fs

In [13]:
print(f'Streaming {T_DUR}s from channel {BEST_CH} at {ap_fs} Hz...')

# Contiguous slice required by remfile — [:, CH:CH+1] not [:, CH]
raw_ap  = ap['data'][:i1, BEST_CH:BEST_CH+1].astype(float)[:, 0] * conv * 1e6  # µV
raw_ap  = raw_ap - np.mean(raw_ap)  # zero-centre

print(f'Done. Shape: {raw_ap.shape}, duration: {t_ap[-1]:.2f}s')
print(f'Signal range: {raw_ap.min():.1f} to {raw_ap.max():.1f} µV')

Streaming 2.0s from channel 60 at 30000.0 Hz...
Done. Shape: (60000,), duration: 2.00s
Signal range: -344.4 to 185.3 µV


## Section : Spectral Analysis

By the end of this tutorial you will learn how to do basic spectral Analysis.

### What is Spectral Analysis?

Spectral analysis is a fundamental technique in signal processing used to understand the frequency content of a signal. Instead of looking at a signal's amplitude changes over time, spectral analysis allows us to see which frequencies are present in the signal and how strong they are. This is particularly useful in fields like neurophysiology  to identify brain rhythms, noise components, or specific neural oscillations.

### How is it done?

The most common method for performing spectral analysis is the **Fast Fourier Transform (FFT)**. The FFT is an efficient algorithm to compute the Discrete Fourier Transform (DFT), which decomposes a signal from its original domain (often time) into a representation in the frequency domain.

Here's a basic overview of the process:

1.  **Time Domain Signal**: We start with a signal recorded over time (e.g., `raw_ap` in our notebook).
2.  **Fast Fourier Transform (FFT)**: We apply the FFT to this time-domain signal. The output of the FFT is a complex-valued array, where each element corresponds to a specific frequency. It essentially tells us the amplitude and phase of each frequency component present in the original signal.
3.  **Frequency Axis (`xf`)**: The `fftfreq` function (from `scipy.fft`) is used to generate the corresponding frequencies for each point in the FFT output.
4.  **Spectrum Calculation**: From the FFT output, we can derive different types of spectra:
    *   **Amplitude Spectrum**: This shows the magnitude (amplitude) of each frequency component. It's calculated as `2.0/N * np.abs(yf[0:N//2])`, where `N` is the number of samples and `yf` is the FFT output. The `2.0/N` scaling normalizes the amplitude, and `[0:N//2]` considers only the positive frequency components (the FFT output is symmetric).
    *   **Power Spectrum**: This shows the power of each frequency component. It's often calculated as the square of the amplitude spectrum, sometimes with an additional normalization (`(1.0/N * np.abs(yf[0:N//2]))**2`). Power spectra are useful because power is additive and directly related to the energy content at each frequency. This is often preferred when comparing the 'strength' of different frequency bands.

By plotting these spectra, we can visually identify dominant frequencies, broadband noise, or other spectral features in the signal.

In [14]:
# Perform FFT on the raw signal
N = len(raw_ap) # Number of sample points
T = 1.0 / ap_fs # Sample spacing (1/sampling frequency)

yf = fft(raw_ap)
xf = fftfreq(N, T)[:N//2]

def combined_interactive_plot(spectrum_type, x_scale, y_scale):
    fig, axes = plt.subplots(2, 1, figsize=(13, 10), sharex=False) # sharex=False as x-axes are different (time vs freq)

    # Top Panel: Raw Signal Over Time
    axes[0].plot(t_ap, raw_ap, color='C4', lw=0.6)
    axes[0].set_title('Raw Broadband Signal Over Time')
    axes[0].set_xlabel('Time (s)')
    axes[0].set_ylabel('Amplitude (µV)')
    axes[0].grid(True, linestyle='--', alpha=0.7)

    # Bottom Panel: Spectrum Plot
    plot_xf = xf
    plot_y_data = None
    y_label = ''

    if spectrum_type == 'Amplitude Spectrum':
        plot_y_data = 2.0/N * np.abs(yf[0:N//2])
        y_label = 'Amplitude (µV)'
    else: # Power Spectrum
        # Using the convention from previous notebook cells for consistency
        plot_y_data = (1.0/N * np.abs(yf[0:N//2]))**2
        y_label = 'Power (µV$^2$)'

    # Exclude DC component (0 Hz) if x-axis is logarithmic to avoid issues with log(0)
    if x_scale == 'log':
        # Find the first non-zero frequency to plot
        first_nonzero_freq_idx = np.where(plot_xf > 0)[0]
        if len(first_nonzero_freq_idx) > 0:
            start_idx = first_nonzero_freq_idx[0]
            plot_xf_log = plot_xf[start_idx:]
            plot_y_data_log = plot_y_data[start_idx:]
        else:
            # Fallback for extremely unusual cases, or if xf is all zeros (shouldn't happen with fftfreq)
            plot_xf_log = np.array([1e-6]) # A small positive value to allow log plot
            plot_y_data_log = np.array([0])
        axes[1].plot(plot_xf_log, plot_y_data_log, color='C1')
    else:
        axes[1].plot(plot_xf, plot_y_data, color='C1')

    axes[1].set_title(f'{spectrum_type} of Raw Broadband Signal')
    axes[1].set_xlabel('Frequency (Hz)')
    axes[1].set_ylabel(y_label)
    axes[1].set_xscale(x_scale)
    axes[1].set_yscale(y_scale)
    axes[1].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    # plt.show() # Removed plt.show() to allow ipywidgets.Output to capture the plot

# Create widgets for user control
spectrum_type_widget = widgets.RadioButtons(
    options=['Amplitude Spectrum', 'Power Spectrum'],
    value='Amplitude Spectrum',
    description='Spectrum Type:',
    disabled=False,
    layout=widgets.Layout(width='auto')
)

x_scale_widget = widgets.RadioButtons(
    options=['linear', 'log'],
    value='linear',
    description='X-axis Scale:',
    disabled=False,
    layout=widgets.Layout(width='auto')
)

y_scale_widget = widgets.RadioButtons(
    options=['linear', 'log'],
    value='linear',
    description='Y-axis Scale:',
    disabled=False,
    layout=widgets.Layout(width='auto')
)

# The interactive_output itself is a widget that captures the plot
output_plot = widgets.interactive_output(combined_interactive_plot, {
    'spectrum_type': spectrum_type_widget,
    'x_scale': x_scale_widget,
    'y_scale': y_scale_widget
})

In [15]:
# @title Arrange the widgets and the plot output in a VBox


In [16]:
# #uncomment to display the wideget
# display(widgets.VBox([
#     output_plot, # The actual plot output from interactive_output
#     widgets.HBox([spectrum_type_widget, x_scale_widget, y_scale_widget]) # Controls second
# ]))

### Visualizing Linear vs. Logarithmic Scales

When visualizing spectral data (amplitude or power spectrum), the choice between linear and logarithmic scales for both the frequency (x-axis) and the amplitude/power (y-axis) can significantly impact what features of the signal become apparent.

#### Y-axis Scale (Amplitude/Power):

*   **Linear Scale**: A linear y-axis emphasizes the strongest components of the signal. If there are a few very dominant frequencies, a linear scale will clearly show their magnitude relative to others. Weaker components might appear as flat lines or be completely obscured by the larger peaks.

    *   **Use case**: Best for signals where you are interested in comparing the absolute magnitudes of large peaks, or when all significant components are within a relatively narrow range of magnitudes.

*   **Logarithmic Scale**: A logarithmic y-axis compresses the range of large values and expands the range of small values. This makes it possible to visualize a wide dynamic range of power or amplitude, bringing out weaker components that would be invisible on a linear scale, while still showing the stronger ones.

    *   **Use case**: Ideal for signals with a large dynamic range, such as broadband noise (which often appears as a relatively flat line on a log scale), or when you want to identify subtle spectral peaks alongside much stronger ones. It's particularly useful in electrophysiology to see both strong neural oscillations and broadband activity.

#### X-axis Scale (Frequency):

*   **Linear Scale**: A linear x-axis provides an even distribution of frequencies across the plot. This is straightforward for inspecting specific frequency ranges.

    *   **Use case**: Useful when you are interested in a specific, narrow range of frequencies, or when you expect evenly spaced frequency components.

*   **Logarithmic Scale**: A logarithmic x-axis (often called a 'log-frequency' plot) compresses higher frequencies and expands lower frequencies. This is often more perceptually relevant for many natural signals and biological data, as changes at lower frequencies tend to be more significant in absolute terms.

    *   **Use case**: Excellent for visualizing signals where frequency content spans many orders of magnitude, or when relative frequency changes are more important than absolute ones. In neuroscience, this is common when observing phenomena across different brain wave bands (e.g., delta, theta, alpha, beta, gamma), which are logarithmically spaced.